# Heart Disease — Preprocessing & Feature Engineering

Ce notebook illustre le pipeline utilisé dans `code/modeling.py` :
`StandardScaler → SMOTE → GridSearchCV`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

## 1. Chargement et nettoyage

In [ ]:
df = pd.read_csv('../data/heart_disease.csv')
print(f'Shape: {df.shape}')
print(f'Doublons: {df.duplicated().sum()}')
print(f'Valeurs manquantes: {df.isnull().sum().sum()}')

df = df.drop_duplicates()

X = df.drop('target', axis=1)
y = df['target']
print(f'\nFeatures: {X.columns.tolist()}')

## 2. Train/Test split (80/20 stratifié)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'X_train: {X_train.shape} | X_test: {X_test.shape}')
print(f'Churn rate — train: {y_train.mean():.3f} | test: {y_test.mean():.3f}')

## 3. StandardScaler

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Shape après scaling — train: {X_train_scaled.shape}')
print(f'Mean features (doit être ~0): {X_train_scaled.mean(axis=0).round(2)}')

## 4. SMOTE — Rééquilibrage

In [ ]:
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_train_scaled, y_train)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
y_train.value_counts().plot(kind='bar', ax=axes[0], color=['#2ecc71', '#e74c3c'])
axes[0].set_title('Avant SMOTE')
axes[0].set_xticklabels(['No Disease', 'Disease'], rotation=0)

pd.Series(y_resampled).value_counts().plot(kind='bar', ax=axes[1], color=['#2ecc71', '#e74c3c'])
axes[1].set_title('Après SMOTE')
axes[1].set_xticklabels(['No Disease', 'Disease'], rotation=0)

plt.suptitle('Impact de SMOTE sur la distribution')
plt.tight_layout()
plt.show()

print(f'Avant: {dict(y_train.value_counts())}')
print(f'Après: {dict(pd.Series(y_resampled).value_counts())}')

## 5. Pipeline complet

Dans `code/modeling.py` :
```python
pipeline = ImbPipeline(steps=[
    ('preprocessor', ColumnTransformer([('num', StandardScaler(), features)])),
    ('smote', SMOTE(random_state=42)),
    ('classifier', model)   # RandomForest / XGBoost / StackingClassifier
])
```
Le meilleur modèle est sauvegardé dans `data/best_model_pipeline.pkl`.